LLM-JEPA
https://github.com/rbalestr-lab/llm-jepa

In [2]:
import yaml
from datasets import load_from_disk
from pathlib import Path
import time

params_path = Path('experiments/llama-32-1b-jepa/params.yaml')

with open(params_path, "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

c:\Users\almei\Documents\GitHub\precision-data\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from train import setup_model_and_tokenizer


model, tokenizer = setup_model_and_tokenizer(
        model_id=params["model_id"],
        use_lora=params["use_lora"],
        lora_rank=params["lora_rank"],
        seed=params["seed"],
        debug=params["debug"],
    )

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


trainable params: 11,272,192 || all params: 1,247,109,120 || trainable%: 0.9039


In [4]:
train_dataset = load_from_disk("datasets/splits-tokenized/train")
val_dataset = load_from_disk("datasets/splits-tokenized/val")

Loading dataset from disk:   0%|          | 0/146 [00:00<?, ?it/s]

In [5]:
from transformers import DataCollatorForLanguageModeling


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We're doing causal LM, not masked LM
    pad_to_multiple_of=None,  # Datasets already padded to max length
)

output_dir = Path(f'experiments/{params["exp_name"]}/outputs')

In [6]:
from transformers import Trainer, TrainingArguments
from transformers.integrations import DVCLiveCallback
from dvclive import Live

from train import ProfilerFLOPCallback, RepresentationTrainer


training_args = TrainingArguments(
    output_dir=output_dir / "checkpoints",
    overwrite_output_dir=True,
    # Training parameters
    per_device_train_batch_size=params["batch_size"],
    per_device_eval_batch_size=params["batch_size"],
    gradient_accumulation_steps=params["gradient_accumulation_steps"],
    learning_rate=params["learning_rate"],
    num_train_epochs=params["num_train_epochs"],
    # Evaluation
    eval_strategy=params["eval_strategy"],  # "steps" if eval_dataset else "no",
    eval_steps=params["eval_steps"],
    # Saving
    save_strategy=params["save_strategy"],
    save_steps=params["save_steps"],
    save_total_limit=params["save_total_limit"],
    # Logging
    logging_dir=output_dir / "logs",
    logging_steps=params["eval_steps"],
    # Optimization - key changes for stability
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,  # Enable for memory efficiency
    dataloader_drop_last=True,  # Drop last incomplete batch
    # Memory optimization
    dataloader_num_workers=0,  # Avoid multiprocessing issues
    # Other
    report_to="dvclive",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    # Disable problematic optimizations
    tf32=False,  # May help with stability
    # Set seed for reproducibility
    seed=params["seed"],
    data_seed=params["seed"],
)

flop_callback = ProfilerFLOPCallback()

if params["regular"]:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        callbacks=[flop_callback] if params["track_flop"] else [],
    )
else:
    trainer = RepresentationTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[flop_callback] if params["track_flop"] else [],
        lbd=params["lbd"],  # Lambda for similarity loss
        gamma=params["gamma"],  # Gamma for LLM loss
        last_token=params["last_token"],  # Index of last token, -1 is '<|eot|>'
        debug=params["debug"],
        additive_mask=params[
            "additive_mask"
        ],  # When set, Use an additive mask to compute both user and assistant in 1 forward pass.
        jepa_l2=params["jepa_l2"],  # When set, Use l2 norm as JEPA loss.
        jepa_mse=params[
            "jepa_mse"
        ],  # When set, Use Mean Squared Error as JEPA loss.
        infonce=params["infonce"],  # When set, Use InfoNCE loss.
        jepa_ratio=params[
            "jepa_ratio"
        ],  # When >0, randomly select this ratio of batches to apply JEPA. This implements Random JEPA-Loss Dropout (LD). If LD = alpha, jepa_ratio = 1 - alpha
    )

trainer.add_callback(
    DVCLiveCallback(Live(dir=Path(f'experiments/{params["exp_name"]}/dvclive')))
)

The model is already on multiple devices. Skipping the move to device specified in `args`.
You are adding a <class 'transformers.integrations.integration_utils.DVCLiveCallback'> to the callbacks of this Trainer, but there is already one. The currentlist of callbacks is
:DefaultFlowCallback
DVCLiveCallback
NotebookProgressCallback


In [ ]:
from tqdm import tqdm
batch_size = 1000
for idx in tqdm(range(len(train_dataset)//batch_size)): 
    try:
        llm_inputs, skip_jepa = trainer.build_with_additive_mask(data_collator(
            [
                train_dataset[i] for i in range(idx*batch_size, (idx+1)*batch_size)
            ]))
    except:
        print('Error at', idx)

  4%|▍         | 97/2331 [08:14<3:07:29,  5.04s/it]

Error at 97


  4%|▍         | 100/2331 [08:26<2:49:49,  4.57s/it]